# Stage 8B-2 — GPU-native RNG + Statistical Parity

### B1 ne yaptı?

Aynı MATLAB random primitives verilince:

\[
H_{BR},H_{RU},F,F_{\rm eff},Y
\]

birebir eşleşti. Yani stochastic matematik kapandı.

### B2 ne yapıyor?

Artık MATLAB random sayılarını Python'a vermiyoruz.

CUDA kendi başına:

\[
XPR,\ ASA,\ ZSA,\ ASD,\ ZSD,\Delta\phi,\Delta\theta,\Phi
\]

üretiyor.

MATLAB ve PyTorch RNG farklı olduğu için:

\[
H^{MATLAB}\neq H^{Python}
\]

olması normal. Bunun yerine dağılım istatistiklerini karşılaştırıyoruz.

Bu aşamada her environment için **tek sabit \(W,z\)** kullanılır.

### B3 ne olacak?

B3 production empirical-label engine olacak:

\[
x
\rightarrow W_{1:K}
\rightarrow z_{1:C}
\rightarrow N_{\rm MC}\ {\rm realizations}
\rightarrow q_{05,\rm emp}
\]

ve ana metriği:

\[
\boxed{\text{empirical q05 labels / second}}
\]

olacak.

Yani B2 = random channel generator doğru dağılımı üretiyor mu?

B3 = bunu milyonlarca \((x,W,z)\) label üretmek için ne kadar hızlı kullanabiliyoruz?

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import sys, zipfile, shutil
import numpy as np
import pandas as pd
import torch

ROOT = Path('/content/drive/MyDrive/MyDrive/RIS')

required = [
    'ris_gpu_physics_stage1.py',
    'ris_gpu_channel_realizations_stage8b1.py',
    'ris_gpu_channel_native_stage8b2.py',
]

for d in [ROOT,Path('/content')]:
    if str(d) not in sys.path:
        sys.path.insert(0,str(d))

missing=[
    f for f in required
    if not (ROOT/f).exists() and not (Path('/content')/f).exists()
]
assert not missing, "Eksik:\n"+"\n".join(missing)

from ris_gpu_channel_native_stage8b2 import run_native_case

device='cuda' if torch.cuda.is_available() else 'cpu'
print("Device:",device)
if torch.cuda.is_available():
    print("GPU:",torch.cuda.get_device_name(0))

## 1. MATLAB reference

MATLAB'da:

```matlab
export_stage8b2_statistical_reference
```

çalıştır.

Default:

```matlab
N_ref = 1000;
```

Bu B2 için yeterli bir distribution sanity testidir. Daha sıkı q05 karşılaştırması
istersek sonra:

```matlab
export_stage8b2_statistical_reference("", "", 4000)
```

yapabiliriz.

Üretilen:

```text
stage8b2_statistical_reference.zip
```

dosyasını `/content` altına yükle.

In [ ]:
ZIP=Path('/content/stage8b2_statistical_reference.zip')
assert ZIP.exists(), "stage8b2_statistical_reference.zip /content altında yok."

EXTRACT=Path('/content/stage8b2_reference_extract')
if EXTRACT.exists():
    shutil.rmtree(EXTRACT)
EXTRACT.mkdir(parents=True)

with zipfile.ZipFile(ZIP,'r') as zf:
    zf.extractall(EXTRACT)

mans=list(EXTRACT.rglob('manifest.csv'))
assert len(mans)==1,mans

SUITE=mans[0].parent
manifest=pd.read_csv(mans[0])

display(manifest)
assert len(manifest)==8

## 2. GPU-native statistical run

Python tarafında reference'dan daha fazla realization kullanıyoruz:

\[
N_{\rm Python}=16384.
\]

Böylece Python distribution tahminindeki Monte-Carlo gürültüsü küçük kalır.
MATLAB reference tarafındaki farkın çoğu \(N_{\rm ref}=1000\) sonlu-örnek
gürültüsünden gelecektir.

In [ ]:
rows=[]

for i,meta in manifest.iterrows():
    print(
        f"[{i+1}/8] {meta['caseName']} | "
        f"nT={meta['nT']} nR={meta['nR']} nRIS={meta['nRIS']}"
    )

    r=run_native_case(
        str(SUITE/str(meta['file'])),
        N_python=16384,
        chunk_size=1024,
        device=device,
        parity=False,
        seed_br=12001+i*101,
        seed_ru=13001+i*101,
    )
    rows.append(r)

df=pd.DataFrame(rows)

In [ ]:
# Link moment/statistical sanity.
link_cols=[
    'scenario','nRIS',
    'BR_meanLeakNorm','BR_meanLeakNorm_ref',
    'BR_varianceRatioMean','BR_varianceRatioMean_ref',
    'BR_fourthMomentRatio','BR_fourthMomentRatio_ref',
    'RU_meanLeakNorm','RU_meanLeakNorm_ref',
    'RU_varianceRatioMean','RU_varianceRatioMean_ref',
    'RU_fourthMomentRatio','RU_fourthMomentRatio_ref',
]

display(df[link_cols])

# Core second/fourth-moment agreement with independent RNG.
core_diff_cols=[
    'BR_varianceRatioMean_relDiff',
    'BR_fourthMomentRatio_relDiff',
    'BR_realVarianceRatio_relDiff',
    'BR_imagVarianceRatio_relDiff',
    'RU_varianceRatioMean_relDiff',
    'RU_fourthMomentRatio_relDiff',
    'RU_realVarianceRatio_relDiff',
    'RU_imagVarianceRatio_relDiff',
]

worst_core=float(df[core_diff_cols].to_numpy().max())
print("Worst link-stat relative difference:",worst_core)

# N_ref=1000 is a statistical reference, not fixed-primitives parity.
assert worst_core < 0.20, (
    f"Link statistical parity suspicious: {worst_core:.3f}"
)

print("PASS: GPU-native link distribution sanity")

In [ ]:
# Cascaded Y distribution / direct empirical q05.
y_cols=[
    'scenario',
    'Y_mean','Y_mean_ref','Y_mean_relDiff',
    'Y_var','Y_var_ref','Y_var_relDiff',
    'Y_q05','Y_q05_ref','Y_q05_relDiff',
    'Y_q50','Y_q50_ref','Y_q50_relDiff',
    'Y_q90','Y_q90_ref','Y_q90_relDiff',
]

display(df[y_cols])

# Tail estimate is noisier because MATLAB reference uses only N_ref samples.
central_cols=[
    'Y_mean_relDiff',
    'Y_var_relDiff',
    'Y_q50_relDiff',
    'Y_q90_relDiff',
]
tail_cols=[
    'Y_q01_relDiff',
    'Y_q05_relDiff',
    'Y_q10_relDiff',
]

worst_central=float(df[central_cols].to_numpy().max())
worst_tail=float(df[tail_cols].to_numpy().max())

print("Worst Y central-stat rel diff:",worst_central)
print("Worst Y tail-quantile rel diff:",worst_tail)

assert worst_central < 0.25
assert worst_tail < 0.35

print("PASS: GPU-native cascaded Y / q05 statistical sanity")

In [ ]:
# Throughput
perf=df[
    [
        'scenario','nT','nR','nRIS','N_python',
        'seconds','realization_pairs_per_second'
    ]
].copy()

display(perf)

print(
    "Median BR+RU stochastic realization-pair throughput:",
    f"{perf['realization_pairs_per_second'].median():,.0f} pair/s"
)

In [ ]:
OUT=Path('/content/stage8b2_statistical_summary.csv')
df.to_csv(OUT,index=False)
print(OUT)

## B2 PASS olursa sıradaki B3

B3'te artık channel correctness test etmeyeceğiz.

Aynı bank/channel state üzerinde:

\[
W_1,\ldots,W_K
\]

ve her \(W_k\) için:

\[
z_1,\ldots,z_C
\]

batchlenecek.

Ama \(F\) tensorünü her \(z\) için devasa biçimde materialize etmek yerine,
doğrudan contraction tasarlayacağız:

\[
F_{\rm eff}^{(n,k,c)}
=
H_{RU}^{(n)}
\operatorname{diag}(\gamma_c)
H_{BR}^{(n)}
W_k.
\]

Sonuçtan doğrudan:

\[
Y_{n,k,c}
=
\|F_{\rm eff}^{(n,k,c)}\|^2
\]

ve:

\[
q_{05,\rm emp}(x,W_k,z_c)
\]

hesaplanacak.

B3'ün başarısı parity değil esas olarak:

\[
\boxed{\text{label/s, GPU memory, q05 convergence}}
\]

ile ölçülecek.